# 📊 Reconciliation 360 Data Model Showcase

This notebook showcases the **dbt data models** built for the Financial Reconciliation Management System.

## Data Architecture
```
Source Tables (COCO_LIVE_DB.CUSTOMER_A_DATA)
    ↓
Staging Models (7 views)
    ↓
Intermediate Models (2 views)
    ↓
Mart Tables (2 tables)
    ↓
Semantic Views (2 semantic views)
```

---

## 1️⃣ Data Volume Overview
Let's see the row counts across our data model layers.

In [ ]:
%%sql -r data_volumes
SELECT 'Source: REC_ASSIGNMENTS' AS layer, COUNT(*) AS row_count FROM COCO_LIVE_DB.CUSTOMER_A_DATA.REC_ASSIGNMENTS
UNION ALL SELECT 'Source: REC_RECONCILIATIONS', COUNT(*) FROM COCO_LIVE_DB.CUSTOMER_A_DATA.REC_RECONCILIATIONS
UNION ALL SELECT 'Source: REC_PERIOD_INFORMATION', COUNT(*) FROM COCO_LIVE_DB.CUSTOMER_A_DATA.REC_PERIOD_INFORMATION
UNION ALL SELECT 'Source: ORG_ENTITIES', COUNT(*) FROM COCO_LIVE_DB.CUSTOMER_A_DATA.ORG_ENTITIES
UNION ALL SELECT 'Mart: ENTITY_360', COUNT(*) FROM COCO_LIVE_DB.DBT_MARTS.ENTITY_360
UNION ALL SELECT 'Mart: ASSIGNMENT_360', COUNT(*) FROM COCO_LIVE_DB.DBT_MARTS.ASSIGNMENT_360
ORDER BY row_count DESC

In [ ]:
import altair as alt
import pandas as pd

alt.data_transformers.enable('vegafusion')

chart = alt.Chart(data_volumes).mark_bar(color='#29B5E8').encode(
    x=alt.X('row_count:Q', title='Row Count', axis=alt.Axis(format='~s')),
    y=alt.Y('layer:N', title='', sort='-x'),
    tooltip=['layer', alt.Tooltip('row_count:Q', format=',')]
).properties(
    title='Data Volume by Model Layer',
    width=600,
    height=250
)
chart

---
## 2️⃣ Entity 360 - Risk Distribution
Analyzing entities by their variance risk level across all periods.

In [ ]:
%%sql -r risk_distribution
SELECT 
    variance_risk_level,
    COUNT(DISTINCT entity_id) AS unique_entities,
    COUNT(*) AS total_records,
    ROUND(AVG(total_variance), 2) AS avg_variance,
    ROUND(SUM(total_variance), 2) AS total_variance
FROM COCO_LIVE_DB.DBT_MARTS.ENTITY_360
GROUP BY variance_risk_level
ORDER BY total_variance DESC

In [ ]:
base = alt.Chart(risk_distribution).encode(
    theta=alt.Theta('total_records:Q', stack=True),
    color=alt.Color('variance_risk_level:N', 
                    scale=alt.Scale(domain=['High', 'Medium', 'Low'], 
                                   range=['#FF6B6B', '#FFE66D', '#4ECDC4']),
                    legend=alt.Legend(title='Risk Level')),
    tooltip=['variance_risk_level', 'unique_entities', alt.Tooltip('total_variance:Q', format=',.0f')]
)

pie = base.mark_arc(outerRadius=100, innerRadius=50).properties(
    title='Entity Records by Risk Level',
    width=300,
    height=300
)

bar = alt.Chart(risk_distribution).mark_bar().encode(
    x=alt.X('variance_risk_level:N', title='Risk Level', sort=['High', 'Medium', 'Low']),
    y=alt.Y('total_variance:Q', title='Total Variance', axis=alt.Axis(format='~s')),
    color=alt.Color('variance_risk_level:N', 
                    scale=alt.Scale(domain=['High', 'Medium', 'Low'], 
                                   range=['#FF6B6B', '#FFE66D', '#4ECDC4']),
                    legend=None),
    tooltip=['variance_risk_level', alt.Tooltip('total_variance:Q', format=',.0f')]
).properties(
    title='Total Variance by Risk Level',
    width=300,
    height=300
)

pie | bar

---
## 3️⃣ Variance Trends Over Time
How does variance change across reconciliation periods?

In [ ]:
%%sql -r period_trends
SELECT 
    period_end_date,
    period_year,
    period_quarter,
    COUNT(DISTINCT entity_id) AS active_entities,
    ROUND(SUM(total_variance), 0) AS total_variance,
    ROUND(SUM(total_unreconciled_amount), 0) AS total_unreconciled,
    ROUND(AVG(assignment_completion_rate), 2) AS avg_completion_rate
FROM COCO_LIVE_DB.DBT_MARTS.ENTITY_360
GROUP BY period_end_date, period_year, period_quarter
ORDER BY period_end_date

In [ ]:
variance_line = alt.Chart(period_trends).mark_line(point=True, color='#FF6B6B').encode(
    x=alt.X('period_end_date:T', title='Period'),
    y=alt.Y('total_variance:Q', title='Total Variance', axis=alt.Axis(format='~s')),
    tooltip=['period_end_date:T', alt.Tooltip('total_variance:Q', format=',.0f')]
).properties(
    title='📈 Total Variance Over Time',
    width=700,
    height=250
)

unreconciled_line = alt.Chart(period_trends).mark_area(opacity=0.6, color='#29B5E8').encode(
    x=alt.X('period_end_date:T', title='Period'),
    y=alt.Y('total_unreconciled:Q', title='Unreconciled Amount', axis=alt.Axis(format='~s')),
    tooltip=['period_end_date:T', alt.Tooltip('total_unreconciled:Q', format=',.0f')]
).properties(
    title='💰 Unreconciled Amounts Over Time',
    width=700,
    height=250
)

variance_line & unreconciled_line

---
## 4️⃣ Assignment 360 - Status Breakdown
Reconciliation status across all assignments.

In [ ]:
%%sql -r status_breakdown
SELECT 
    reconciliation_status,
    variance_risk,
    COUNT(DISTINCT assignment_id) AS assignment_count,
    ROUND(SUM(unreconciled_amount), 0) AS total_unreconciled,
    ROUND(AVG(total_variance), 0) AS avg_variance
FROM COCO_LIVE_DB.DBT_MARTS.ASSIGNMENT_360
GROUP BY reconciliation_status, variance_risk
ORDER BY assignment_count DESC

In [ ]:
heatmap = alt.Chart(status_breakdown).mark_rect().encode(
    x=alt.X('reconciliation_status:N', title='Reconciliation Status'),
    y=alt.Y('variance_risk:N', title='Variance Risk', sort=['High', 'Medium', 'Low']),
    color=alt.Color('assignment_count:Q', 
                    scale=alt.Scale(scheme='blues'),
                    title='Assignment Count'),
    tooltip=['reconciliation_status', 'variance_risk', 
             alt.Tooltip('assignment_count:Q', format=','),
             alt.Tooltip('total_unreconciled:Q', format=',.0f')]
).properties(
    title='🔥 Status vs Risk Heatmap',
    width=500,
    height=200
)

status_bar = alt.Chart(status_breakdown).mark_bar().encode(
    x=alt.X('reconciliation_status:N', title='Status'),
    y=alt.Y('sum(assignment_count):Q', title='Assignments'),
    color=alt.Color('variance_risk:N',
                   scale=alt.Scale(domain=['High', 'Medium', 'Low'], 
                                  range=['#FF6B6B', '#FFE66D', '#4ECDC4']),
                   title='Risk'),
    tooltip=['reconciliation_status', 'variance_risk', alt.Tooltip('assignment_count:Q', format=',')]
).properties(
    title='📊 Assignments by Status & Risk',
    width=500,
    height=250
)

heatmap & status_bar

---
## 5️⃣ Top Entities by Variance
Which entities have the highest total variance across all periods?

In [ ]:
%%sql -r top_entities
SELECT 
    entity_name,
    entity_code,
    COUNT(DISTINCT period_id) AS periods_tracked,
    ROUND(SUM(total_variance), 0) AS total_variance,
    ROUND(AVG(assignment_completion_rate), 1) AS avg_completion_rate,
    MAX(variance_risk_level) AS max_risk_level
FROM COCO_LIVE_DB.DBT_MARTS.ENTITY_360
GROUP BY entity_name, entity_code
HAVING SUM(total_variance) > 0
ORDER BY total_variance DESC
LIMIT 15

In [ ]:
entity_bar = alt.Chart(top_entities).mark_bar(color='#667eea').encode(
    x=alt.X('total_variance:Q', title='Total Variance', axis=alt.Axis(format='~s')),
    y=alt.Y('entity_name:N', title='', sort='-x'),
    color=alt.Color('max_risk_level:N',
                   scale=alt.Scale(domain=['High', 'Medium', 'Low'], 
                                  range=['#FF6B6B', '#FFE66D', '#4ECDC4']),
                   title='Max Risk'),
    tooltip=['entity_name', 'entity_code', 
             alt.Tooltip('total_variance:Q', format=',.0f'),
             alt.Tooltip('avg_completion_rate:Q', format='.1f'),
             'max_risk_level']
).properties(
    title='🏆 Top 15 Entities by Total Variance',
    width=600,
    height=400
)

entity_bar

---
## 6️⃣ Assignment Types Distribution
Breakdown of assignment types (Balance, Flux, Activity, etc.)

In [ ]:
%%sql -r assignment_types
SELECT 
    assignment_type,
    COUNT(DISTINCT assignment_id) AS assignments,
    COUNT(DISTINCT entity_id) AS entities,
    ROUND(SUM(balance_gl), 0) AS total_gl_balance,
    ROUND(SUM(total_variance), 0) AS total_variance,
    ROUND(AVG(total_variance), 0) AS avg_variance
FROM COCO_LIVE_DB.DBT_MARTS.ASSIGNMENT_360
WHERE assignment_type IS NOT NULL
GROUP BY assignment_type
ORDER BY assignments DESC